# Freddie Mac Multi-Year Origination Validation

## Purpose

This notebook applies the reusable Freddie Mac origination pipeline to the
2015–2019 annual samples.

The detailed 2015 EDA established the baseline cleaning and validation
decisions. This notebook checks whether those decisions remain appropriate
for subsequent years and identifies:

- structural or schema problems;
- undocumented categorical codes;
- documented sentinel values;
- changes in missingness and feature ranges;
- disclosure or data-definition changes;
- potential borrower-population changes requiring later drift analysis.

The notebook initially validates 2015 and 2016 and will be extended through
2019 after each annual sample is reviewed.

In [1]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display


# Identify the repository root when the notebook runs from /notebooks.
CURRENT_DIRECTORY = Path.cwd()

if CURRENT_DIRECTORY.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIRECTORY.parent
else:
    PROJECT_ROOT = CURRENT_DIRECTORY

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


from src.freddie_ingestion import load_origination
from src.freddie_quality import create_origination_quality_report


print("Notebook directory:", CURRENT_DIRECTORY)
print("Project root:", PROJECT_ROOT)

Notebook directory: e:\capstone_project\mortgage-model-reliability\notebooks
Project root: e:\capstone_project\mortgage-model-reliability


## 1. File Configuration

Define the annual origination files using paths relative to the project root.
The notebook will stop if an expected input file is missing.

In [ ]:
YEARS = [2015, 2016, 2017, 2018, 2019]

RAW_DATA_DIRECTORY = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "Freddie_Mac"
)

ORIGINATION_FILES = {
    year: (
        RAW_DATA_DIRECTORY
        / f"sample_{year}"
        / f"sample_orig_{year}.txt"
    )
    for year in YEARS
}


for year, file_path in ORIGINATION_FILES.items():
    print(
        year,
        "| Exists:",
        file_path.exists(),
        "| Path:",
        file_path,
    )

2015 | Exists: True | Path: e:\capstone_project\mortgage-model-reliability\data\raw\Freddie_Mac\sample_2015\sample_orig_2015.txt
2016 | Exists: True | Path: e:\capstone_project\mortgage-model-reliability\data\raw\Freddie_Mac\sample_2016\sample_orig_2016.txt
2017 | Exists: True | Path: e:\capstone_project\mortgage-model-reliability\data\raw\Freddie_Mac\sample_2017\sample_orig_2017.txt
2018 | Exists: True | Path: e:\capstone_project\mortgage-model-reliability\data\raw\Freddie_Mac\sample_2018\sample_orig_2018.txt


## 2. Load and Structurally Validate Annual Origination Data

`load_origination()` first validates the raw file structure and then loads the
pipe-delimited records into a pandas DataFrame. At this stage, no cleaning is
performed.

In [3]:
raw_origination = {}
structure_rows = []

for year, file_path in ORIGINATION_FILES.items():
    annual_data = load_origination(file_path)

    raw_origination[year] = annual_data

    structure_report = annual_data.attrs[
        "structure_validation"
    ]

    structure_rows.append(
        {
            "year": year,
            **structure_report,
        }
    )


structure_validation = pd.DataFrame(structure_rows)

display(structure_validation)

,year,rows_scanned,expected_fields_per_row,field_count_errors,field_length_errors
0,2015,50000,31,0,0
1,2016,50000,31,0,0
2,2017,50000,31,0,0
3,2018,50000,31,0,0


### Multi-Year Quality Report Generation

Generate the standardized quality reports for every year listed in `YEARS`.
Store each annual report by year and summarize the number of undocumented
categorical codes and unsupported high-ratio records.

In [4]:
quality_reports = {}
quality_check_rows = []

for year in YEARS:
    annual_quality_report = (
        create_origination_quality_report(
            data=raw_origination[year],
            year=year,
        )
    )

    quality_reports[year] = annual_quality_report

    quality_check_rows.append(
        {
            "year": year,
            "unexpected_code_rows": len(
                annual_quality_report[
                    "unexpected_codes"
                ]
            ),
            "unsupported_high_ratio_records": len(
                annual_quality_report[
                    "high_ratio_warnings"
                ]
            ),
        }
    )


quality_check_summary = pd.DataFrame(
    quality_check_rows
)


display(quality_check_summary)

,year,unexpected_code_rows,unsupported_high_ratio_records
0,2015,0,0
1,2016,0,0
2,2017,0,0
3,2018,0,0


In [5]:
for year, data in raw_origination.items():
    print(
        year,
        "| Rows:",
        data.shape[0],
        "| Columns:",
        data.shape[1],
    )

2015 | Rows: 50000 | Columns: 31
2016 | Rows: 50000 | Columns: 31
2017 | Rows: 50000 | Columns: 31
2018 | Rows: 50000 | Columns: 31


## 3. Annual Data-Quality Reports

The quality checks do not modify the raw data. They report:

- undocumented categorical codes;
- documented sentinel frequencies;
- raw missingness by feature.

Unexpected codes require investigation before cleaning or interpreting
year-to-year differences as borrower-population drift.

In [6]:
quality_reports = {}

for year, data in raw_origination.items():
    quality_reports[year] = (
        create_origination_quality_report(
            data=data,
            year=year,
        )
    )


unexpected_summary = pd.DataFrame(
    [
        {
            "year": year,
            "unexpected_code_rows": len(
                quality_reports[year]["unexpected_codes"]
            ),
        }
        for year in YEARS
    ]
)

display(unexpected_summary)

,year,unexpected_code_rows
0,2015,0
1,2016,0
2,2017,0
3,2018,0


### 3.1 Documented Sentinel Frequencies

Sentinels are documented codes representing unavailable, unknown, or
undisclosed information. Differences in sentinel frequency may reflect data
availability or disclosure changes rather than borrower-population drift.

In [7]:
sentinel_counts = pd.concat(
    [
        quality_reports[year]["sentinel_counts"]
        for year in YEARS
    ],
    ignore_index=True,
)


nonzero_sentinels = (
    sentinel_counts.loc[
        sentinel_counts["count"] > 0,
        [
            "year",
            "column",
            "sentinel",
            "count",
            "percentage",
        ],
    ]
    .sort_values(
        by=["column", "year"]
    )
    .reset_index(drop=True)
)


display(nonzero_sentinels)

,year,column,sentinel,count,percentage
0,2017,CLASSIC FICO,9999,8,0.016
1,2018,CLASSIC FICO,9999,15,0.030
2,2015,FIRST TIME HOMEBUYER INDICATOR,9,8,0.016
3,2016,FIRST TIME HOMEBUYER INDICATOR,9,2,0.004
4,2015,ORIGINAL COMBINED LOAN-TO-VALUE (CLTV),999,1,0.002
5,2018,ORIGINAL COMBINED LOAN-TO-VALUE (CLTV),999,1,0.002
6,2015,ORIGINAL DEBT-TO-INCOME (DTI) RATIO,999,4020,8.040
7,2016,ORIGINAL DEBT-TO-INCOME (DTI) RATIO,999,2528,5.056
8,2017,ORIGINAL DEBT-TO-INCOME (DTI) RATIO,999,1806,3.612
9,2018,ORIGINAL DEBT-TO-INCOME (DTI) RATIO,999,513,1.026


### 3.2 Property Valuation Method Availability Across Years

In [8]:
property_valuation_rows = []

for year in YEARS:
    property_valuation_method = raw_origination[
        year
    ]["PROPERTY VALUATION METHOD"]

    value_counts = (
        property_valuation_method
        .value_counts(dropna=False)
        .sort_index()
    )

    for value, count in value_counts.items():
        property_valuation_rows.append(
            {
                "year": year,
                "property_valuation_method": value,
                "count": int(count),
                "percentage": (
                    count
                    / len(property_valuation_method)
                    * 100
                ),
            }
        )


property_valuation_comparison = pd.DataFrame(
    property_valuation_rows
)


display(
    property_valuation_comparison.round(3)
)

,year,property_valuation_method,count,percentage
0,2015,7,50000,100.000
1,2016,7,50000,100.000
2,2017,1,445,0.890
3,2017,7,49555,99.110
4,2018,1,1513,3.026
5,2018,7,48487,96.974


### Property Valuation Method Findings

- Property Valuation Method was completely unavailable in the 2015 and 2016
  samples: all 50,000 loans in each year contained documented code `7`.
- Actual valuation information begins appearing in the 2017 sample.
- In 2017, 445 loans (0.89%) contain code `1`, while 49,555 loans (99.11%)
  retain code `7`.
- Freddie Mac defines code `1` as an appraisal waiver under its Automated
  Collateral Evaluation (ACE) program.
- Freddie Mac defines code `7` as Property Valuation Method not available.
- The limited 2017 availability is consistent with the transition to Property
  Valuation Method disclosure for securities issued on or after June 1, 2017.
- Code `7` should continue to be converted to missing in the corresponding
  `_CLEAN` column. It should not be treated as an actual valuation method.
- The 445 usable 2017 observations should be retained, but the field remains
  too sparsely populated in 2017 for reliable standalone modeling.
- Availability should continue to be monitored in the 2018 and 2019 samples.

Source: [Freddie Mac Property Valuation Method disclosure documentation](https://capitalmarkets.freddiemac.com/mbs/docs/appraisal_waiver_disclosure_announcement.pdf)



### 3.3 Raw Missingness Comparison

Compare raw null percentages across every year listed in `YEARS`. Documented
sentinel values are reported separately and are not counted as raw missing
values. Percentage-point changes are calculated automatically between each
pair of consecutive years.

In [9]:
missingness = pd.concat(
    [
        quality_reports[year]["missingness"]
        for year in YEARS
    ],
    ignore_index=True,
)


missingness_comparison = (
    missingness.pivot(
        index="column",
        columns="year",
        values="missing_percentage",
    )
    .reset_index()
)

missingness_comparison.columns.name = None


missing_column_names = {
    year: f"missing_{year}_pct"
    for year in YEARS
}

missingness_comparison = (
    missingness_comparison.rename(
        columns=missing_column_names
    )
)


for previous_year, current_year in zip(
    YEARS[:-1],
    YEARS[1:],
):
    previous_column = (
        f"missing_{previous_year}_pct"
    )

    current_column = (
        f"missing_{current_year}_pct"
    )

    change_column = (
        f"change_{current_year}"
        f"_minus_{previous_year}_pp"
    )

    missingness_comparison[
        change_column
    ] = (
        missingness_comparison[current_column]
        - missingness_comparison[previous_column]
    )


annual_missingness_columns = [
    f"missing_{year}_pct"
    for year in YEARS
]


nonzero_missingness = (
    missingness_comparison.loc[
        missingness_comparison[
            annual_missingness_columns
        ]
        .gt(0)
        .any(axis=1)
    ]
    .sort_values("column")
    .reset_index(drop=True)
)


display(
    nonzero_missingness.round(3)
)

,column,missing_2015_pct,missing_2016_pct,missing_2017_pct,missing_2018_pct,change_2016_minus_2015_pp,change_2017_minus_2016_pp,change_2018_minus_2017_pp
0,METROPOLITAN STATISTICAL AREA (MSA) OR METROPO...,11.208,10.108,10.804,10.340,-1.100,0.696,-0.464
1,PRE-HARP LOAN SEQUENCE NUMBER,91.966,94.950,96.392,98.978,2.984,1.442,2.586
2,SPECIAL ELIGIBILITY PROGRAM,99.350,97.884,93.022,88.662,-1.466,-4.862,-4.360


### Raw Missingness Findings

- Only three raw fields contain null values across the 2015–2017 samples:
  MSA, Pre-HARP Loan Sequence Number, and Special Eligibility Program.
- MSA missingness decreased from 11.208% in 2015 to 10.108% in 2016, then
  increased slightly to 10.804% in 2017.
- Pre-HARP Loan Sequence Number missingness increased from 91.966% in 2015
  to 94.950% in 2016 and 96.392% in 2017.
- The increasing Pre-HARP missingness is consistent with the shrinking share
  of HARP-related loans across these annual samples.
- Special Eligibility Program missingness decreased from 99.350% in 2015 to
  97.884% in 2016 and 93.022% in 2017.
- The 4.862 percentage-point decline from 2016 to 2017 indicates that a
  larger share of 2017 loans contains a Special Eligibility Program value.
- These are raw null percentages. Documented sentinel values are evaluated
  separately and should not be interpreted as ordinary missing values.
- The observed changes should be retained for later population-drift
  analysis rather than treated as ingestion errors.

In [10]:
from src.freddie_cleaning import clean_origination
from src.freddie_validation import validate_origination

## 4. Cleaning and Critical Validation

Cleaning preserves every raw field and appends `_CLEAN` fields. Critical
validation stops processing if the schema, row count, loan identifiers,
cleaned sentinels, or postal-code representation fail.

In [11]:
cleaned_origination = {}
validation_rows = []

for year, raw_data in raw_origination.items():
    cleaned_data = clean_origination(raw_data)

    validation_report = validate_origination(
        raw=raw_data,
        cleaned=cleaned_data,
        expected_rows=50000,
    )

    cleaned_origination[year] = cleaned_data

    validation_rows.append(
        {
            "year": year,
            **validation_report,
        }
    )


validation_summary = pd.DataFrame(validation_rows)

display(validation_summary)

,year,rows,raw_columns,cleaned_columns,missing_loan_ids,duplicate_loan_ids,invalid_postal_codes
0,2015,50000,31,47,0,0,0
1,2016,50000,31,47,0,0,0
2,2017,50000,31,47,0,0,0
3,2018,50000,31,47,0,0,0


### Cleaning and Validation Findings

- The reusable cleaning pipeline successfully processed the 2015, 2016, and
  2017 annual samples.
- Each year contains 50,000 rows and the expected 31 raw columns.
- Cleaning preserved every raw field and added six corresponding clean
  fields, producing 37 columns.
- No rows were added or removed during cleaning.
- No missing or duplicate Loan Identifiers were detected.
- All cleaned postal codes contain the required three-digit representation.
- The 2017 sample therefore passes the same critical cleaning and validation
  requirements established for the 2015 baseline and confirmed for 2016.

In [12]:
from src.freddie_summary import create_annual_summary

## 5. Standardized Annual Summaries

The same summary function is applied to every year so that differences are
comparable. Clean fields are used where documented sentinels were replaced
with missing values.

In [13]:
annual_summaries = {}

for year, cleaned_data in cleaned_origination.items():
    annual_summaries[year] = create_annual_summary(
        data=cleaned_data,
        year=year,
    )


numeric_summary = pd.concat(
    [
        annual_summaries[year]["numeric"]
        for year in YEARS
    ],
    ignore_index=True,
)

categorical_summary = pd.concat(
    [
        annual_summaries[year]["categorical"]
        for year in YEARS
    ],
    ignore_index=True,
)


display(
    numeric_summary.round(2)
)

,year,variable,count,missing,missing_percentage,mean,std,min,q1,median,q3,max
0,2015,FICO,50000.0,0,0.00,749.76,47.51,462.00,718.00,760.00,788.00,834.00
1,2015,DTI,45980.0,4020,8.04,33.84,9.39,1.00,27.00,35.00,42.00,50.00
2,2015,LTV,49999.0,1,0.00,73.51,17.51,5.00,65.00,78.00,83.00,241.00
3,2015,CLTV,49999.0,1,0.00,74.40,18.02,5.00,66.00,79.00,85.00,854.00
4,2015,UPB,50000.0,0,0.00,224006.06,119990.50,10000.00,131000.00,200000.00,299000.00,974000.00
5,2015,INTEREST RATE,50000.0,0,0.00,3.97,0.45,2.38,3.75,4.00,4.25,5.50
6,2015,LOAN TERM,50000.0,0,0.00,315.29,75.92,96.00,240.00,360.00,360.00,366.00
7,2015,MI PERCENTAGE,50000.0,0,0.00,5.90,11.15,0.00,0.00,0.00,0.00,40.00
8,2015,VANTAGESCORE 4.0,0.0,50000,100.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,2016,FICO,50000.0,0,0.00,748.79,46.71,456.00,716.00,757.00,787.00,832.00


### 5.1 Multi-Year High-Ratio Comparison

Apply the same high-ratio definitions across every available year. Compare
the number and percentage of loans with LTV or CLTV above 105%, verify their
HARP and Pre-HARP support, and monitor changes in the maximum observed ratio
values.

In [14]:
high_ratio_summary_rows = []

for year, data in cleaned_origination.items():
    annual_ltv = data[
        "ORIGINAL LOAN-TO-VALUE (LTV)_CLEAN"
    ]

    annual_cltv = data[
        "ORIGINAL COMBINED LOAN-TO-VALUE (CLTV)_CLEAN"
    ]

    high_ratio_mask = (
        annual_ltv.gt(105)
        | annual_cltv.gt(105)
    )

    high_ratio_records = data.loc[high_ratio_mask]

    high_ratio_summary_rows.append(
        {
            "year": year,
            "high_ratio_records": len(
                high_ratio_records
            ),
            "high_ratio_percentage": (
                len(high_ratio_records)
                / len(data)
                * 100
            ),
            "ltv_above_105": int(
                annual_ltv.gt(105).sum()
            ),
            "cltv_above_105": int(
                annual_cltv.gt(105).sum()
            ),
            "harp_y": int(
                high_ratio_records[
                    "HARP INDICATOR"
                ].eq("Y").sum()
            ),
            "pre_harp_id_present": int(
                high_ratio_records[
                    "PRE-HARP LOAN SEQUENCE NUMBER"
                ].notna().sum()
            ),
            "maximum_ltv": annual_ltv.max(),
            "maximum_cltv": annual_cltv.max(),
        }
    )


high_ratio_summary = pd.DataFrame(
    high_ratio_summary_rows
)

display(
    high_ratio_summary.round(3)
)

,year,high_ratio_records,high_ratio_percentage,ltv_above_105,cltv_above_105,harp_y,pre_harp_id_present,maximum_ltv,maximum_cltv
0,2015,530,1.060,361,530,530,530,241.0,854.0
1,2016,227,0.454,148,227,227,227,336.0,336.0
2,2017,124,0.248,90,124,124,124,194.0,194.0
3,2018,34,0.068,24,34,34,34,222.0,255.0


### 5.2 Investigate Extreme 2016 LTV and CLTV Values

The central LTV and CLTV distributions are stable, but the maximum value of
336 requires record-level investigation. High values are retained until they
can be classified as valid special-program loans, documented disclosure
values, or data-quality anomalies.

In [15]:
ltv_2016 = cleaned_origination[2016][
    "ORIGINAL LOAN-TO-VALUE (LTV)_CLEAN"
]

cltv_2016 = cleaned_origination[2016][
    "ORIGINAL COMBINED LOAN-TO-VALUE (CLTV)_CLEAN"
]


print("2016 LTV > 105:", int(ltv_2016.gt(105).sum()))
print("2016 CLTV > 105:", int(cltv_2016.gt(105).sum()))
print("2016 LTV = 336:", int(ltv_2016.eq(336).sum()))
print("2016 CLTV = 336:", int(cltv_2016.eq(336).sum()))

2016 LTV > 105: 148
2016 CLTV > 105: 227
2016 LTV = 336: 1
2016 CLTV = 336: 1


In [16]:
extreme_ratio_records_2016 = (
    cleaned_origination[2016]
    .loc[
        ltv_2016.gt(105) | cltv_2016.gt(105),
        [
            "LOAN IDENTIFIER",
            "ORIGINAL LOAN-TO-VALUE (LTV)_CLEAN",
            "ORIGINAL COMBINED LOAN-TO-VALUE (CLTV)_CLEAN",
            "LOAN PURPOSE",
            "SUPER CONFORMING FLAG",
            "PRE-HARP LOAN SEQUENCE NUMBER",
            "SPECIAL ELIGIBILITY PROGRAM",
            "HARP INDICATOR",
            "ORIGINAL UPB",
            "PROPERTY TYPE",
            "OCCUPANCY STATUS",
        ],
    ]
    .copy()
)


display(
    extreme_ratio_records_2016
    .sort_values(
        by=[
            "ORIGINAL LOAN-TO-VALUE (LTV)_CLEAN",
            "ORIGINAL COMBINED LOAN-TO-VALUE (CLTV)_CLEAN",
        ],
        ascending=False,
    )
    .head(20)
)

,LOAN IDENTIFIER,ORIGINAL LOAN-TO-VALUE (LTV)_CLEAN,ORIGINAL COMBINED LOAN-TO-VALUE (CLTV)_CLEAN,LOAN PURPOSE,SUPER CONFORMING FLAG,PRE-HARP LOAN SEQUENCE NUMBER,SPECIAL ELIGIBILITY PROGRAM,HARP INDICATOR,ORIGINAL UPB,PROPERTY TYPE,OCCUPANCY STATUS
37415,F16Q30476204,336,336,N,N,A08Q30002418,<NA>,Y,138000,CO,P
8090,F16Q10200259,260,333,N,N,F07Q10243093,<NA>,Y,125000,CO,I
24334,F16Q20409731,222,222,N,N,F09Q20311057,<NA>,Y,334000,SF,P
11179,F16Q10275314,183,183,N,N,F06Q40393669,<NA>,Y,155000,SF,P
19277,F16Q20236155,175,186,N,N,F06Q40317266,<NA>,Y,76000,SF,P
3568,F16Q10089992,161,161,N,N,F06Q30152221,<NA>,Y,67000,SF,P
11807,F16Q10291861,155,170,N,N,F07Q30152312,<NA>,Y,212000,SF,P
9162,F16Q10226493,155,155,N,N,F07Q20027043,<NA>,Y,294000,SF,P
11880,F16Q10293693,155,155,N,N,F07Q20020434,<NA>,Y,96000,SF,P
23178,F16Q20369946,153,153,N,N,F08Q10465752,<NA>,Y,123000,CO,P


In [17]:
high_ratio_mask_2016 = (
    ltv_2016.gt(105)
    | cltv_2016.gt(105)
)

high_ratio_data_2016 = cleaned_origination[2016].loc[
    high_ratio_mask_2016
]


print(
    "Total records with LTV or CLTV > 105:",
    len(high_ratio_data_2016),
)

print(
    "LTV > 105:",
    int(ltv_2016.gt(105).sum()),
)

print(
    "CLTV > 105:",
    int(cltv_2016.gt(105).sum()),
)

print(
    "HARP = Y:",
    int(
        high_ratio_data_2016[
            "HARP INDICATOR"
        ].eq("Y").sum()
    ),
)

print(
    "Pre-HARP sequence present:",
    int(
        high_ratio_data_2016[
            "PRE-HARP LOAN SEQUENCE NUMBER"
        ].notna().sum()
    ),
)

print()
print("Loan-purpose counts:")
print(
    high_ratio_data_2016[
        "LOAN PURPOSE"
    ].value_counts(dropna=False)
)

Total records with LTV or CLTV > 105: 227
LTV > 105: 148
CLTV > 105: 227
HARP = Y: 227
Pre-HARP sequence present: 227

Loan-purpose counts:
LOAN PURPOSE
N    227
Name: count, dtype: int64[pyarrow]


### Extreme-ratio findings for 2016

- A total of 227 loans have LTV or CLTV above 105%.
- All 227 loans are identified as HARP loans.
- All 227 have a populated Pre-HARP Loan Sequence Number.
- All 227 have no-cash-out refinance loan purpose (`N`).
- The maximum LTV and CLTV of 336% are consistent with the special
  high-LTV refinance population represented by HARP.
- These values are preserved and must not be treated as generic range
  errors or replaced with missing values.
- Future range validation should evaluate high ratios together with their
  program indicators rather than applying a universal 105% upper limit.

### 5.3 Investigate Extreme 2017 LTV and CLTV Values

The multi-year high-ratio summary identified 124 loans in 2017 with LTV or
CLTV above 105%, with maximum LTV and CLTV values of 194%. Investigate these
records to confirm whether they are supported by HARP and Pre-HARP
information and whether their other loan characteristics are consistent with
the special refinance population.

In [18]:
ltv_2017 = cleaned_origination[2017][
    "ORIGINAL LOAN-TO-VALUE (LTV)_CLEAN"
]

cltv_2017 = cleaned_origination[2017][
    "ORIGINAL COMBINED LOAN-TO-VALUE (CLTV)_CLEAN"
]


high_ratio_mask_2017 = (
    ltv_2017.gt(105)
    | cltv_2017.gt(105)
)


high_ratio_data_2017 = (
    cleaned_origination[2017]
    .loc[high_ratio_mask_2017]
    .copy()
)


pre_harp_2017 = (
    high_ratio_data_2017[
        "PRE-HARP LOAN SEQUENCE NUMBER"
    ]
    .astype("string")
    .str.strip()
)


pre_harp_present_2017 = (
    pre_harp_2017.notna()
    & pre_harp_2017.ne("")
)


print(
    "Total records with LTV or CLTV > 105:",
    len(high_ratio_data_2017),
)

print(
    "LTV > 105:",
    int(ltv_2017.gt(105).sum()),
)

print(
    "CLTV > 105:",
    int(cltv_2017.gt(105).sum()),
)

print(
    "Maximum LTV:",
    ltv_2017.max(),
)

print(
    "Maximum CLTV:",
    cltv_2017.max(),
)

print(
    "HARP = Y:",
    int(
        high_ratio_data_2017[
            "HARP INDICATOR"
        ].eq("Y").sum()
    ),
)

print(
    "Pre-HARP sequence present:",
    int(pre_harp_present_2017.sum()),
)

print(
    "Unsupported high-ratio warnings:",
    len(
        quality_reports[2017][
            "high_ratio_warnings"
        ]
    ),
)

print()
print("Loan-purpose counts:")

print(
    high_ratio_data_2017[
        "LOAN PURPOSE"
    ].value_counts(dropna=False)
)


extreme_ratio_records_2017 = (
    high_ratio_data_2017.loc[
        :,
        [
            "LOAN IDENTIFIER",
            "ORIGINAL LOAN-TO-VALUE (LTV)_CLEAN",
            "ORIGINAL COMBINED LOAN-TO-VALUE (CLTV)_CLEAN",
            "LOAN PURPOSE",
            "HARP INDICATOR",
            "PRE-HARP LOAN SEQUENCE NUMBER",
            "SPECIAL ELIGIBILITY PROGRAM",
            "PROPERTY VALUATION METHOD",
            "PROPERTY VALUATION METHOD_CLEAN",
            "ORIGINAL UPB",
            "PROPERTY TYPE",
            "OCCUPANCY STATUS",
        ],
    ]
    .sort_values(
        by=[
            "ORIGINAL LOAN-TO-VALUE (LTV)_CLEAN",
            "ORIGINAL COMBINED LOAN-TO-VALUE (CLTV)_CLEAN",
        ],
        ascending=False,
    )
    .reset_index(drop=True)
)


display(
    extreme_ratio_records_2017.head(20)
)

Total records with LTV or CLTV > 105: 124
LTV > 105: 90
CLTV > 105: 124
Maximum LTV: 194
Maximum CLTV: 194
HARP = Y: 124
Pre-HARP sequence present: 124
Unsupported high-ratio warnings: 0

Loan-purpose counts:
LOAN PURPOSE
N    124
Name: count, dtype: int64[pyarrow]


,LOAN IDENTIFIER,ORIGINAL LOAN-TO-VALUE (LTV)_CLEAN,ORIGINAL COMBINED LOAN-TO-VALUE (CLTV)_CLEAN,LOAN PURPOSE,HARP INDICATOR,PRE-HARP LOAN SEQUENCE NUMBER,SPECIAL ELIGIBILITY PROGRAM,PROPERTY VALUATION METHOD,PROPERTY VALUATION METHOD_CLEAN,ORIGINAL UPB,PROPERTY TYPE,OCCUPANCY STATUS
0,F17Q10258526,194,194,N,Y,A07Q10022117,<NA>,7,NaN,119000,CO,P
1,F17Q20323918,175,175,N,Y,F06Q30382564,<NA>,7,NaN,33000,CO,I
2,F17Q20282716,170,170,N,Y,A06Q30065689,<NA>,7,NaN,85000,CO,I
3,F17Q10100616,162,162,N,Y,F05Q20424707,<NA>,7,NaN,94000,SF,I
4,F17Q30132977,158,158,N,Y,F06Q40407951,<NA>,7,NaN,131000,MH,P
5,F17Q10080085,151,151,N,Y,A08Q10022320,<NA>,7,NaN,232000,SF,I
6,F17Q20326457,150,150,N,Y,F07Q20029366,<NA>,7,NaN,90000,CO,P
7,F17Q10171629,149,149,N,Y,F06Q30162259,<NA>,7,NaN,103000,CO,I
8,F17Q10247304,147,147,N,Y,F06Q30217093,<NA>,7,NaN,107000,CO,I
9,F17Q10157563,146,168,N,Y,A06Q40054717,<NA>,7,NaN,192000,SF,P


### Extreme-Ratio Findings for 2017

- A total of 124 loans (0.248% of the annual sample) have LTV or CLTV above
  105%.
- Of these, 90 loans have LTV above 105%, while all 124 have CLTV above
  105%. The high-LTV loans are therefore contained within the high-CLTV
  population.
- The maximum observed LTV and CLTV are both 194%.
- All 124 high-ratio loans have `HARP INDICATOR = Y`.
- All 124 also contain a populated Pre-HARP Loan Sequence Number.
- The automated program-aware quality rule produced zero unsupported
  high-ratio warnings.
- The 20 highest-ratio records all have no-cash-out refinance purpose (`N`),
  which is consistent with the HARP refinance population.
- The highest-ratio records cover multiple property and occupancy types,
  rather than being isolated to one property segment.
- Property Valuation Method is `7` for all 20 displayed records and is
  correctly converted to missing in `PROPERTY VALUATION METHOD_CLEAN`.
- Special Eligibility Program is also missing for the displayed records;
  HARP status is instead supported directly by the HARP indicator and
  Pre-HARP loan linkage.
- The 2017 high-ratio values are retained because the associated records
  contain consistent program evidence.
- The high-ratio population continued to decline from 530 loans in 2015 to
  227 in 2016 and 124 in 2017, indicating a shrinking HARP-related segment
  rather than a data-quality problem.

### High-ratio comparison findings

- All high-ratio records in both years are linked to HARP and contain a
  Pre-HARP Loan Sequence Number.
- The high-ratio population decreased from 530 loans (1.060%) in 2015 to
  227 loans (0.454%) in 2016.
- This represents a 0.606 percentage-point reduction in the annual sample's
  HARP-related high-ratio segment.
- The change represents product/program mix variation rather than malformed
  data or a schema difference.
- The 2015 maximum CLTV of 854 and the 2016 maximum LTV/CLTV of 336 are
  retained because their records have consistent HARP indicators.
- The reusable quality pipeline now flags LTV or CLTV above 105% when the
  record is not supported by both a HARP indicator and a populated Pre-HARP
  Loan Sequence Number.
- Applying this rule produced zero warnings for both 2015 and 2016.

In [19]:
for year in YEARS:
    high_ratio_warnings = quality_reports[
        year
    ]["high_ratio_warnings"]

    print(
        year,
        "| Unsupported high-ratio records:",
        len(high_ratio_warnings),
    )

    display(high_ratio_warnings)

2015 | Unsupported high-ratio records: 0


,year,origination_quarter,LOAN IDENTIFIER,ORIGINAL LOAN-TO-VALUE (LTV),ORIGINAL COMBINED LOAN-TO-VALUE (CLTV),HARP INDICATOR,PRE-HARP LOAN SEQUENCE NUMBER,HIGH LTV FLAG,HIGH CLTV FLAG


2016 | Unsupported high-ratio records: 0


,year,origination_quarter,LOAN IDENTIFIER,ORIGINAL LOAN-TO-VALUE (LTV),ORIGINAL COMBINED LOAN-TO-VALUE (CLTV),HARP INDICATOR,PRE-HARP LOAN SEQUENCE NUMBER,HIGH LTV FLAG,HIGH CLTV FLAG


2017 | Unsupported high-ratio records: 0


,year,origination_quarter,LOAN IDENTIFIER,ORIGINAL LOAN-TO-VALUE (LTV),ORIGINAL COMBINED LOAN-TO-VALUE (CLTV),HARP INDICATOR,PRE-HARP LOAN SEQUENCE NUMBER,HIGH LTV FLAG,HIGH CLTV FLAG


2018 | Unsupported high-ratio records: 0


,year,origination_quarter,LOAN IDENTIFIER,ORIGINAL LOAN-TO-VALUE (LTV),ORIGINAL COMBINED LOAN-TO-VALUE (CLTV),HARP INDICATOR,PRE-HARP LOAN SEQUENCE NUMBER,HIGH LTV FLAG,HIGH CLTV FLAG


Before connecting this to the pipeline, confirm the rule can actually detect an unsupported record the rule can actually detect an unsupported record

In [27]:
quality_rule_test_rows = []
quality_rule_warning_tables = []


for year in YEARS:
    quality_rule_test = (
        raw_origination[year]
        .iloc[[0, 1, 2]]
        .copy()
    )

    year_suffix = str(year)[-2:]

    quality_rule_test[
        "LOAN IDENTIFIER"
    ] = [
        f"T{year_suffix}Q10000001",
        f"T{year_suffix}Q20000002",
        f"T{year_suffix}Q20000003",
    ]

    quality_rule_test[
        "ORIGINAL LOAN-TO-VALUE (LTV)"
    ] = [
        120,
        120,
        1000,
    ]

    quality_rule_test[
        "ORIGINAL COMBINED LOAN-TO-VALUE (CLTV)"
    ] = [
        120,
        120,
        1000,
    ]

    quality_rule_test[
        "HARP INDICATOR"
    ] = "N"

    quality_rule_test[
        "PRE-HARP LOAN SEQUENCE NUMBER"
    ] = pd.NA


    test_quality_report = (
        create_origination_quality_report(
            data=quality_rule_test,
            year=year,
        )
    )

    detected_warnings = test_quality_report[
        "high_ratio_warnings"
    ]

    if year < 2018:
        expected_warning_count = 3

    elif year == 2018:
        expected_warning_count = 2

    else:
        expected_warning_count = 1


    quality_rule_test_rows.append(
        {
            "year": year,
            "expected_warnings": (
                expected_warning_count
            ),
            "detected_warnings": len(
                detected_warnings
            ),
            "test_passed": (
                len(detected_warnings)
                == expected_warning_count
            ),
        }
    )

    quality_rule_warning_tables.append(
        detected_warnings
    )


quality_rule_test_results = pd.DataFrame(
    quality_rule_test_rows
)


quality_rule_test_warnings = pd.concat(
    quality_rule_warning_tables,
    ignore_index=True,
)


display(quality_rule_test_results)

display(quality_rule_test_warnings)

assert quality_rule_test_results[
    "test_passed"
].all()




,year,expected_warnings,detected_warnings,test_passed
0,2015,3,3,True
1,2016,3,3,True
2,2017,3,3,True
3,2018,2,2,True


,year,origination_quarter,LOAN IDENTIFIER,ORIGINAL LOAN-TO-VALUE (LTV),ORIGINAL COMBINED LOAN-TO-VALUE (CLTV),HARP INDICATOR,PRE-HARP LOAN SEQUENCE NUMBER,HIGH LTV FLAG,HIGH CLTV FLAG
0,2015,1,T15Q10000001,120,120,N,<NA>,True,False
1,2015,2,T15Q20000002,120,120,N,<NA>,True,False
2,2015,2,T15Q20000003,1000,1000,N,<NA>,True,True
3,2016,1,T16Q10000001,120,120,N,<NA>,True,False
4,2016,2,T16Q20000002,120,120,N,<NA>,True,False
5,2016,2,T16Q20000003,1000,1000,N,<NA>,True,True
6,2017,1,T17Q10000001,120,120,N,<NA>,True,False
7,2017,2,T17Q20000002,120,120,N,<NA>,True,False
8,2017,2,T17Q20000003,1000,1000,N,<NA>,True,True
9,2018,1,T18Q10000001,120,120,N,<NA>,True,False


## 6. Categorical Distribution Comparison

Category percentages are aligned across years. The largest percentage-point
changes are reviewed as possible population or product-mix changes, not as
formal statistical drift conclusions.

In [21]:
categorical_comparison = (
    categorical_summary.pivot_table(
        index=[
            "variable",
            "category",
        ],
        columns="year",
        values="percentage",
        fill_value=0,
    )
    .reset_index()
)

categorical_comparison.columns.name = None


percentage_column_names = {
    year: f"percentage_{year}"
    for year in YEARS
}

categorical_comparison = (
    categorical_comparison.rename(
        columns=percentage_column_names
    )
)


categorical_change_columns = []


for previous_year, current_year in zip(
    YEARS[:-1],
    YEARS[1:],
):
    previous_column = (
        f"percentage_{previous_year}"
    )

    current_column = (
        f"percentage_{current_year}"
    )

    change_column = (
        f"change_{current_year}"
        f"_minus_{previous_year}_pp"
    )

    categorical_comparison[
        change_column
    ] = (
        categorical_comparison[current_column]
        - categorical_comparison[previous_column]
    )

    categorical_change_columns.append(
        change_column
    )


categorical_comparison[
    "maximum_absolute_change_pp"
] = (
    categorical_comparison[
        categorical_change_columns
    ]
    .abs()
    .max(axis=1)
)


largest_categorical_changes = (
    categorical_comparison
    .sort_values(
        "maximum_absolute_change_pp",
        ascending=False,
    )
    .head(20)
    .reset_index(drop=True)
)


display(
    largest_categorical_changes.round(3)
)

,variable,category,percentage_2015,percentage_2016,percentage_2017,percentage_2018,change_2016_minus_2015_pp,change_2017_minus_2016_pp,change_2018_minus_2017_pp,maximum_absolute_change_pp
0,LOAN PURPOSE,N,35.026,33.478,20.140,12.298,-1.548,-13.338,-7.842,13.338
1,LOAN PURPOSE,P,44.308,43.944,57.064,66.826,-0.364,13.120,9.762,13.120
2,FIRST TIME HOMEBUYER,Y,15.580,16.088,21.472,26.690,0.508,5.384,5.218,5.384
3,FIRST TIME HOMEBUYER,N,84.404,83.908,78.528,73.310,-0.496,-5.380,-5.218,5.380
4,PROPERTY STATE,CA,15.576,15.984,12.956,10.036,0.408,-3.028,-2.920,3.028
5,CHANNEL,R,60.860,60.894,59.346,56.676,0.034,-1.548,-2.670,2.670
6,OCCUPANCY STATUS,P,88.192,88.844,86.530,87.338,0.652,-2.314,0.808,2.314
7,NUMBER OF BORROWERS GROUP,2+,51.080,49.436,47.476,45.878,-1.644,-1.960,-1.598,1.960
8,NUMBER OF BORROWERS GROUP,1,48.920,50.564,52.524,54.122,1.644,1.960,1.598,1.960
9,CHANNEL,C,29.232,29.308,31.240,33.148,0.076,1.932,1.908,1.932


In [22]:
print(
    largest_categorical_changes
    .round(3)
    .to_dict(orient="records")
)

[{'variable': 'LOAN PURPOSE', 'category': 'N', 'percentage_2015': 35.026, 'percentage_2016': 33.478, 'percentage_2017': 20.14, 'percentage_2018': 12.298, 'change_2016_minus_2015_pp': -1.548, 'change_2017_minus_2016_pp': -13.338, 'change_2018_minus_2017_pp': -7.842, 'maximum_absolute_change_pp': 13.338}, {'variable': 'LOAN PURPOSE', 'category': 'P', 'percentage_2015': 44.308, 'percentage_2016': 43.944, 'percentage_2017': 57.064, 'percentage_2018': 66.826, 'change_2016_minus_2015_pp': -0.364, 'change_2017_minus_2016_pp': 13.12, 'change_2018_minus_2017_pp': 9.762, 'maximum_absolute_change_pp': 13.12}, {'variable': 'FIRST TIME HOMEBUYER', 'category': 'Y', 'percentage_2015': 15.58, 'percentage_2016': 16.088, 'percentage_2017': 21.472, 'percentage_2018': 26.69, 'change_2016_minus_2015_pp': 0.508, 'change_2017_minus_2016_pp': 5.384, 'change_2018_minus_2017_pp': 5.218, 'maximum_absolute_change_pp': 5.384}, {'variable': 'FIRST TIME HOMEBUYER', 'category': 'N', 'percentage_2015': 84.404, 'percen

### Categorical Distribution Findings

- Categorical distributions were generally stable between 2015 and 2016, but
  several larger composition changes appeared between 2016 and 2017.
- Loan Purpose shows the largest change:
  - No-cash-out refinance (`N`) decreased from 33.478% in 2016 to 20.140% in
    2017, a decline of 13.338 percentage points.
  - Purchase loans (`P`) increased from 43.944% to 57.064%, an increase of
    13.120 percentage points.
  - Cash-out refinance (`C`) remained comparatively stable, increasing by only
    0.218 percentage points between 2016 and 2017.
- First-time homebuyers increased from 16.088% in 2016 to 21.472% in 2017,
  while non-first-time homebuyers declined correspondingly.
- California's share decreased by 3.028 percentage points between 2016 and
  2017, while Texas and Florida increased by 0.822 and 0.798 percentage points,
  respectively.
- Primary-residence loans decreased by 2.314 percentage points, while
  investment-property loans increased by 1.784 percentage points.
- Single-borrower loans increased by 1.960 percentage points, while
  two-borrower loans decreased by the same amount.
- Correspondent-channel loans increased by 1.932 percentage points between
  2016 and 2017, while retail-channel loans decreased by 1.548 percentage
  points.
- Property-type distributions remained comparatively stable. The largest
  listed change was a 0.574 percentage-point decline in single-family
  properties between 2015 and 2016.
- These results are descriptive comparisons of annual sample composition. They
  identify candidate population and product-mix changes but are not, by
  themselves, formal statistical evidence of drift.
- The large 2017 changes in Loan Purpose and first-time-homebuyer composition
  should be examined further during the formal drift-analysis phase.

In [23]:
from src.freddie_pipeline import run_origination_pipeline

## 7. Save Verified Annual Outputs

After the annual quality findings have been reviewed, run the end-to-end
pipeline for every year listed in `YEARS`. Save each cleaned loan-level dataset
together with its validation, numeric summary, categorical summary, sentinel,
missingness, and quality-report outputs.

In [24]:
PROCESSED_DATA_DIRECTORY = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "Freddie_Mac"
)


pipeline_results = {}


for year in YEARS:
    pipeline_result = run_origination_pipeline(
        year=year,
        input_file=ORIGINATION_FILES[year],
        output_directory=(
            PROCESSED_DATA_DIRECTORY
        ),
        expected_rows=50000,
    )

    pipeline_results[year] = pipeline_result


    print(f"Year: {year}")
    print()

    print("Structure validation:")

    print(
        pipeline_result[
            "structure_validation"
        ]
    )

    print()

    print("Data validation:")

    print(
        pipeline_result[
            "validation"
        ]
    )

    print()

    print("Saved outputs:")

    for output_name, output_path in (
        pipeline_result[
            "output_paths"
        ].items()
    ):
        print(
            f"{output_name}: {output_path}"
        )

    print()
    print("=" * 70)
    print()

Year: 2015

Structure validation:
{'rows_scanned': 50000, 'expected_fields_per_row': 31, 'field_count_errors': 0, 'field_length_errors': 0}

Data validation:
{'rows': 50000, 'raw_columns': 31, 'cleaned_columns': 47, 'missing_loan_ids': 0, 'duplicate_loan_ids': 0, 'invalid_postal_codes': 0}

Saved outputs:
cleaned: e:\capstone_project\mortgage-model-reliability\data\processed\Freddie_Mac\origination_2015_clean.parquet
numeric_summary: e:\capstone_project\mortgage-model-reliability\data\processed\Freddie_Mac\origination_2015_numeric_summary.csv
categorical_summary: e:\capstone_project\mortgage-model-reliability\data\processed\Freddie_Mac\origination_2015_categorical_summary.csv
unexpected_codes: e:\capstone_project\mortgage-model-reliability\data\processed\Freddie_Mac\origination_2015_unexpected_codes.csv
sentinel_counts: e:\capstone_project\mortgage-model-reliability\data\processed\Freddie_Mac\origination_2015_sentinel_counts.csv
missingness: e:\capstone_project\mortgage-model-reliabili

### Saved Annual Output Findings

- The reusable end-to-end pipeline completed successfully for 2015, 2016, and
  2017.
- Each annual raw file contained exactly 50,000 rows with the expected 31
  fields per row.
- No field-count or field-length errors were detected.
- Cleaning preserved all 50,000 rows and 31 raw columns while adding six clean
  columns, producing 37 columns.
- No missing or duplicate Loan Identifiers were detected.
- No invalid cleaned postal codes were detected.
- For each year, the pipeline saved:
  - The cleaned loan-level dataset in Parquet format.
  - Numeric and categorical summaries.
  - Unexpected-code results.
  - Sentinel counts.
  - Raw missingness results.
  - Unsupported high-ratio warnings.
  - A validation report in JSON format.
- The same workflow will automatically process additional years after they are
  added to `YEARS` and `ORIGINATION_FILES`.

Read back the saved high-ratio warning CSV and validation JSON for every
available year. Confirm that the program-aware quality results were written
correctly and that the warning counts agree across both saved outputs.

In [25]:
import json


saved_high_ratio_warnings = {}
saved_validation_reports = {}
saved_output_check_rows = []


for year in YEARS:
    annual_pipeline_result = (
        pipeline_results[year]
    )


    annual_saved_warnings = pd.read_csv(
        annual_pipeline_result[
            "output_paths"
        ]["high_ratio_warnings"]
    )


    with annual_pipeline_result[
        "output_paths"
    ]["validation"].open(
        mode="r",
        encoding="utf-8",
    ) as validation_file:
        annual_saved_validation = json.load(
            validation_file
        )


    saved_high_ratio_warnings[
        year
    ] = annual_saved_warnings

    saved_validation_reports[
        year
    ] = annual_saved_validation


    csv_warning_rows = len(
        annual_saved_warnings
    )

    json_warning_rows = (
        annual_saved_validation[
            "high_ratio_warning_rows"
        ]
    )


    saved_output_check_rows.append(
        {
            "year": year,
            "saved_csv_warning_rows": (
                csv_warning_rows
            ),
            "validation_json_warning_rows": (
                json_warning_rows
            ),
            "warning_counts_match": (
                csv_warning_rows
                == json_warning_rows
            ),
        }
    )


saved_output_check = pd.DataFrame(
    saved_output_check_rows
)


display(saved_output_check)


assert (
    saved_output_check[
        "warning_counts_match"
    ].all()
), "Saved warning counts do not match."

,year,saved_csv_warning_rows,validation_json_warning_rows,warning_counts_match
0,2015,0,0,True
1,2016,0,0,True
2,2017,0,0,True
3,2018,0,0,True


## 8. Multi-Year Saved-Dataset Verification

Read back each saved Parquet dataset and validate it again to confirm that
serialization preserved the expected rows, columns, cleaned fields, loan
identifiers, and postal-code representation for every available year.

In [26]:
saved_cleaned_origination = {}
readback_validations = {}


for year in YEARS:
    saved_cleaned_data = pd.read_parquet(
        pipeline_results[year][
            "output_paths"
        ]["cleaned"]
    )


    readback_validation = validate_origination(
        raw=raw_origination[year],
        cleaned=saved_cleaned_data,
        expected_rows=50000,
    )


    saved_cleaned_origination[
        year
    ] = saved_cleaned_data

    readback_validations[
        year
    ] = readback_validation


    print(f"Year: {year}")

    print(
        "Saved shape:",
        saved_cleaned_data.shape,
    )

    print(
        "Postal-code dtype:",
        saved_cleaned_data[
            "POSTAL CODE_CLEAN"
        ].dtype,
    )

    print(
        "Read-back validation:",
        readback_validation,
    )

    print()
    print("=" * 70)
    print()

Year: 2015
Saved shape: (50000, 47)
Postal-code dtype: string
Read-back validation: {'rows': 50000, 'raw_columns': 31, 'cleaned_columns': 47, 'missing_loan_ids': 0, 'duplicate_loan_ids': 0, 'invalid_postal_codes': 0}


Year: 2016
Saved shape: (50000, 47)
Postal-code dtype: string
Read-back validation: {'rows': 50000, 'raw_columns': 31, 'cleaned_columns': 47, 'missing_loan_ids': 0, 'duplicate_loan_ids': 0, 'invalid_postal_codes': 0}


Year: 2017
Saved shape: (50000, 47)
Postal-code dtype: string
Read-back validation: {'rows': 50000, 'raw_columns': 31, 'cleaned_columns': 47, 'missing_loan_ids': 0, 'duplicate_loan_ids': 0, 'invalid_postal_codes': 0}


Year: 2018
Saved shape: (50000, 47)
Postal-code dtype: string
Read-back validation: {'rows': 50000, 'raw_columns': 31, 'cleaned_columns': 47, 'missing_loan_ids': 0, 'duplicate_loan_ids': 0, 'invalid_postal_codes': 0}




### Multi-Year Read-Back Findings

- The saved 2015, 2016, and 2017 Parquet datasets were read back successfully.
- Each saved dataset contains 50,000 rows and 37 columns.
- All 31 original raw columns and six added clean columns were preserved.
- `POSTAL CODE_CLEAN` retained the pandas `string` data type, preserving
  three-digit postal-code prefixes and leading zeros.
- No missing or duplicate Loan Identifiers were detected after serialization.
- No invalid cleaned postal codes were detected.
- The saved datasets therefore satisfy the same critical validation requirements
  as the in-memory cleaned datasets.

## 9. Conclusion

The reusable Freddie Mac origination pipeline was successfully applied to the
2015, 2016, and 2017 annual samples.

### Validation Outcome

- Each annual sample contains 50,000 records, giving 150,000 processed records
  across the three years.
- Every raw record contains the expected 31 fields.
- No field-count or maximum-length violations were detected.
- No undocumented categorical codes were detected.
- Cleaning preserved all 31 raw fields and appended six clean fields, producing
  37 columns per annual dataset.
- No missing or duplicate Loan Identifiers were detected.
- No invalid cleaned postal-code values were detected.
- Every saved Parquet dataset was successfully read back with 50,000 rows and
  37 columns.
- The saved postal-code fields retained the pandas `string` data type.
- The program-aware high-ratio quality rule produced zero warnings for all
  three actual annual samples.
- A year-by-year rule-verification test confirmed that the quality rule detects
  unsupported high-ratio records when HARP evidence is deliberately removed.
- The saved CSV and JSON high-ratio warning counts agree for every year.

### Multi-Year Observations

- DTI sentinel usage decreased from 8.040% in 2015 to 5.056% in 2016 and
  3.612% in 2017.
- Property Valuation Method was completely unavailable in 2015 and 2016.
  Usable valuation-method information began appearing in 2017, although only
  445 loans (0.89%) contained code `1`.
- Raw null values were limited to MSA, Pre-HARP Loan Sequence Number, and
  Special Eligibility Program.
- The HARP-related high-ratio population declined from 530 loans (1.060%) in
  2015 to 227 loans (0.454%) in 2016 and 124 loans (0.248%) in 2017.
- Every actual loan with LTV or CLTV above 105% was supported by both
  `HARP INDICATOR = Y` and a populated Pre-HARP Loan Sequence Number.
- The maximum observed ratios declined from a maximum CLTV of 854% in 2015 to
  336% in 2016 and 194% in 2017.
- Categorical distributions were relatively stable between 2015 and 2016.
- A larger product-mix change appeared in 2017:
  - No-cash-out refinance loans decreased by 13.338 percentage points.
  - Purchase loans increased by 13.120 percentage points.
  - First-time homebuyers increased by 5.384 percentage points.
- The declining no-cash-out refinance share is consistent with the shrinking
  HARP-related segment.

All three annual samples pass the current ingestion, cleaning, quality, and
validation requirements. The observed annual differences identify candidate
population and product-mix changes, but they are not yet formal evidence of
statistical drift or model degradation. These questions will be evaluated in
the later drift and model-reliability stages.